## Desafio Sprint-8 Spark

In [ ]:
# instalando as bibliotecas necessárias
!pip install -q pyspark
!pip install -q findspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
# Importando as bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark import SparkContext, SQLContext
from pyspark.sql.functions import col, split, lit, regexp_replace, rand, when, udf, isnull, size, monotonically_increasing_id, lower, trim
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import random

In [ ]:
# iniciando sessão Spark
spark = SparkSession.builder.appName("mySparkSession").getOrCreate()

In [ ]:
!unzip Filmes+e+Series_backup.zip

Archive:  Filmes+e+Series_backup.zip
  inflating: movies.csv              
  inflating: series.csv              


## Trabalhando com a união dos dois DF csv

In [ ]:
# Baixando o arquivo e lendo as primeiras linhas
df_movie_uniao = spark.read.csv("/content/movies.csv", header= True, inferSchema=True, sep="|")
df_serie_uniao = spark.read.csv("/content/series.csv", header= True, inferSchema=True, sep="|")
df_movie_uniao.show(5)
df_serie_uniao.show(5)

+---------+--------------+--------------+-------------+------------+-------+---------+-----------+-------------+--------------------+-----------------+-------------+--------------+------------+---------------------+
|       id|tituloPincipal|tituloOriginal|anoLancamento|tempoMinutos| genero|notaMedia|numeroVotos|generoArtista|          personagem|      nomeArtista|anoNascimento|anoFalecimento|   profissao|titulosMaisConhecidos|
+---------+--------------+--------------+-------------+------------+-------+---------+-----------+-------------+--------------------+-----------------+-------------+--------------+------------+---------------------+
|tt0000009|    Miss Jerry|    Miss Jerry|         1894|          45|Romance|      5.3|        200|      actress|Miss Geraldine Ho...|  Blanche Bayliss|         1878|          1951|     actress|            tt0000009|
|tt0000009|    Miss Jerry|    Miss Jerry|         1894|          45|Romance|      5.3|        200|        actor|        Mr. Hamilton|Wil

### Conferindo os esquemas

In [ ]:
# imprimindo os schemas das tabelas
df_movie_uniao.printSchema()
df_serie_uniao.printSchema()

root
 |-- id: string (nullable = true)
 |-- tituloPincipal: string (nullable = true)
 |-- tituloOriginal: string (nullable = true)
 |-- anoLancamento: string (nullable = true)
 |-- tempoMinutos: string (nullable = true)
 |-- genero: string (nullable = true)
 |-- notaMedia: double (nullable = true)
 |-- numeroVotos: integer (nullable = true)
 |-- generoArtista: string (nullable = true)
 |-- personagem: string (nullable = true)
 |-- nomeArtista: string (nullable = true)
 |-- anoNascimento: string (nullable = true)
 |-- anoFalecimento: string (nullable = true)
 |-- profissao: string (nullable = true)
 |-- titulosMaisConhecidos: string (nullable = true)

root
 |-- id: string (nullable = true)
 |-- tituloPincipal: string (nullable = true)
 |-- tituloOriginal: string (nullable = true)
 |-- anoLancamento: string (nullable = true)
 |-- anoTermino: string (nullable = true)
 |-- tempoMinutos: string (nullable = true)
 |-- genero: string (nullable = true)
 |-- notaMedia: double (nullable = true)


In [ ]:
# Casting da coluna 'tempoMinutos' para o tipo integer
df_movie_uniao = df_movie_uniao.withColumn('tempoMinutos', df_movie_uniao['tempoMinutos'].cast('int'))
df_serie_uniao = df_serie_uniao.withColumn('tempoMinutos', df_serie_uniao['tempoMinutos'].cast('int'))

In [ ]:
# imprimindo os schemas das tabelas
df_movie_uniao.printSchema()
df_serie_uniao.printSchema()

root
 |-- id: string (nullable = true)
 |-- tituloPincipal: string (nullable = true)
 |-- tituloOriginal: string (nullable = true)
 |-- anoLancamento: string (nullable = true)
 |-- tempoMinutos: integer (nullable = true)
 |-- genero: string (nullable = true)
 |-- notaMedia: double (nullable = true)
 |-- numeroVotos: integer (nullable = true)
 |-- generoArtista: string (nullable = true)
 |-- personagem: string (nullable = true)
 |-- nomeArtista: string (nullable = true)
 |-- anoNascimento: string (nullable = true)
 |-- anoFalecimento: string (nullable = true)
 |-- profissao: string (nullable = true)
 |-- titulosMaisConhecidos: string (nullable = true)

root
 |-- id: string (nullable = true)
 |-- tituloPincipal: string (nullable = true)
 |-- tituloOriginal: string (nullable = true)
 |-- anoLancamento: string (nullable = true)
 |-- anoTermino: string (nullable = true)
 |-- tempoMinutos: integer (nullable = true)
 |-- genero: string (nullable = true)
 |-- notaMedia: double (nullable = true

### Criando uma coluna chamada 'tipoMidia' em ambas os df

In [ ]:
df_movie_uniao = df_movie_uniao.withColumn('tipoMidia', lit('movie'))
df_serie_uniao = df_serie_uniao.withColumn('tipoMidia', lit('serie'))
df_movie_uniao.show(5)
df_serie_uniao.show(5)

+---------+--------------+--------------+-------------+------------+-------+---------+-----------+-------------+--------------------+-----------------+-------------+--------------+------------+---------------------+---------+
|       id|tituloPincipal|tituloOriginal|anoLancamento|tempoMinutos| genero|notaMedia|numeroVotos|generoArtista|          personagem|      nomeArtista|anoNascimento|anoFalecimento|   profissao|titulosMaisConhecidos|tipoMidia|
+---------+--------------+--------------+-------------+------------+-------+---------+-----------+-------------+--------------------+-----------------+-------------+--------------+------------+---------------------+---------+
|tt0000009|    Miss Jerry|    Miss Jerry|         1894|          45|Romance|      5.3|        200|      actress|Miss Geraldine Ho...|  Blanche Bayliss|         1878|          1951|     actress|            tt0000009|    movie|
|tt0000009|    Miss Jerry|    Miss Jerry|         1894|          45|Romance|      5.3|        20

### Unindo os datasets

In [ ]:
df_movie_series = df_movie_uniao.unionByName(df_serie_uniao, allowMissingColumns=True)

# Preenchendo os valores nulos com 'não contêm'
df_movie_series = df_movie_series.fillna('sem informação')

df_movie_series.show(5)

+---------+--------------+--------------+-------------+------------+-------+---------+-----------+-------------+--------------------+-----------------+-------------+--------------+------------+---------------------+---------+--------------+
|       id|tituloPincipal|tituloOriginal|anoLancamento|tempoMinutos| genero|notaMedia|numeroVotos|generoArtista|          personagem|      nomeArtista|anoNascimento|anoFalecimento|   profissao|titulosMaisConhecidos|tipoMidia|    anoTermino|
+---------+--------------+--------------+-------------+------------+-------+---------+-----------+-------------+--------------------+-----------------+-------------+--------------+------------+---------------------+---------+--------------+
|tt0000009|    Miss Jerry|    Miss Jerry|         1894|          45|Romance|      5.3|        200|      actress|Miss Geraldine Ho...|  Blanche Bayliss|         1878|          1951|     actress|            tt0000009|    movie|sem informação|
|tt0000009|    Miss Jerry|    Miss J

In [ ]:
quantidade_de_linhas = df_movie_series.count()
print(f"O DataFrame tem {quantidade_de_linhas} linhas.")

O DataFrame tem 1513944 linhas.


### Fazendo a limpeza do DF

In [ ]:
# No caso quero trabalhar com artistas ainda vivos, pois o meu json tem avaliação
# de artistas que ainda estão trabalhando, assim também quero selecionar atores que
# tenham até 80 anos.

# Também estou eliminando as linhas onde não há o genero do filme e selecionando
# filmes com até 50 anos de lançamento e por fim estou eliminando a coluna
# "titulosMaisConhecidos", pois não vai ser usada.

df_movie_series = df_movie_series[(df_movie_series["anoFalecimento"] == '\\N') &
                                  (df_movie_series["genero"] != '\\N') &
                                  (df_movie_series["anoNascimento"] >= 1944) &
                                  (df_movie_series["anoLancamento"] >= 1974)].drop("titulosMaisConhecidos") # Remove the 'columns' argument and just pass the column name as a string

df_movie_series.show(5)

+---------+--------------+--------------+-------------+------------+--------------------+---------+-----------+-------------+-------------+-----------------+-------------+--------------+--------------------+---------+--------------+
|       id|tituloPincipal|tituloOriginal|anoLancamento|tempoMinutos|              genero|notaMedia|numeroVotos|generoArtista|   personagem|      nomeArtista|anoNascimento|anoFalecimento|           profissao|tipoMidia|    anoTermino|
+---------+--------------+--------------+-------------+------------+--------------------+---------+-----------+-------------+-------------+-----------------+-------------+--------------+--------------------+---------+--------------+
|tt0015724| Dama de noche| Dama de noche|         1993|         102|Drama,Mystery,Rom...|      6.1|         26|      actress|       Salomé|    Regina Orozco|         1964|            \N|             actress|    movie|sem informação|
|tt0015724| Dama de noche| Dama de noche|         1993|         102|

In [ ]:
quantidade_de_linhas = df_movie_series.count()
print(f"O DataFrame tem {quantidade_de_linhas} linhas.")

O DataFrame tem 521187 linhas.


In [ ]:
# Selecionando as linhas com pelo menos um dos valores "war" ou "crime"
df_movie_series = df_movie_series.filter((col("genero").contains("War") | col("genero").contains("Crime")))
# Garantido que teremos no df somente pessoas que são atores e atrizes
df_movie_series = df_movie_series.filter((col("profissao").contains("actress") | col("profissao").contains("actor")))
df_movie_series.show(truncate=False)

+---------+---------------------------------------------+------------------------------------------------+-------------+------------+----------------------+---------+-----------+-------------+--------------------+--------------------+-------------+--------------+------------------------------------+---------+--------------+
|id       |tituloPincipal                               |tituloOriginal                                  |anoLancamento|tempoMinutos|genero                |notaMedia|numeroVotos|generoArtista|personagem          |nomeArtista         |anoNascimento|anoFalecimento|profissao                           |tipoMidia|anoTermino    |
+---------+---------------------------------------------+------------------------------------------------+-------------+------------+----------------------+---------+-----------+-------------+--------------------+--------------------+-------------+--------------+------------------------------------+---------+--------------+
|tt0036606|Another Tim

In [ ]:
quantidade_de_linhas = df_movie_series.count()
print(f"O DataFrame tem {quantidade_de_linhas} linhas.")

O DataFrame tem 63088 linhas.


In [ ]:
# Removendo colunas desnecessárias
colunas_para_remover = ['anoFalecimento']
df_movie_series = df_movie_series.drop(*colunas_para_remover)

# Modificando o valor da coluna 'generoArtista' para valores numericos
df_movie_series = df_movie_series.withColumn("generoArtista",
                                            when(df_movie_series["generoArtista"] == "actor", 2).otherwise(1))
df_movie_series.show(5)

+---------+---------+--------------------+--------------------+-------------+--------------+------------+--------------------+---------+-----------+-------------+--------------------+---------------+-------------+--------------------+--------------+
|       id|tipoMidia|      tituloPincipal|      tituloOriginal|anoLancamento|    anoTermino|tempoMinutos|              genero|notaMedia|numeroVotos|generoArtista|          personagem|    nomeArtista|anoNascimento|           profissao|        decada|
+---------+---------+--------------------+--------------------+-------------+--------------+------------+--------------------+---------+-----------+-------------+--------------------+---------------+-------------+--------------------+--------------+
|tt0071327|    movie|        Ciemna rzeka|        Ciemna rzeka|         1974|sem informação|         101|           Drama,War|      5.3|         19|            1|Wladka Stankiewic...|  Wanda Neumann|         1945|             actress|decada_setenta|


In [ ]:
# Limpando valores nulos
df_movie_series = df_movie_series.na.fill({'tempoMinutos': 0})

In [ ]:
# Limpando valores nulos
@udf(returnType=StringType())
def replace_nan(value):
    if value == '\\N':
        return 'sem informação'
    else:
        return value

# Aplicando a função às colunas necessárias
df_movie_series = df_movie_series.withColumn("tituloPincipal", replace_nan(df_movie_series["tituloPincipal"]))
df_movie_series = df_movie_series.withColumn("anoLancamento", replace_nan(df_movie_series["anoLancamento"]))
df_movie_series = df_movie_series.withColumn("anoTermino", replace_nan(df_movie_series["anoTermino"]))
df_movie_series = df_movie_series.withColumn("personagem", replace_nan(df_movie_series["personagem"]))

In [ ]:
df_movie_series.show(5)

+---------+--------------------+--------------------+-------------+------------+------------------+---------+-----------+-------------+--------------+--------------------+-------------+--------------------+---------+--------------+
|       id|      tituloPincipal|      tituloOriginal|anoLancamento|tempoMinutos|            genero|notaMedia|numeroVotos|generoArtista|    personagem|         nomeArtista|anoNascimento|           profissao|tipoMidia|    anoTermino|
+---------+--------------------+--------------------+-------------+------------+------------------+---------+-----------+-------------+--------------+--------------------+-------------+--------------------+---------+--------------+
|tt0036606|Another Time, Ano...|Another Time, Ano...|         1983|         118|         Drama,War|      6.5|        327|            2|         Luigi|  Giovanni Mauriello|         1945|actor,music_depar...|    movie|sem informação|
|tt0036606|Another Time, Ano...|Another Time, Ano...|         1983|     

In [ ]:
# Colocando as colunas na ordem desejada
df_movie_series = df_movie_series.select('id', 'tipoMidia', 'tituloPincipal', 'tituloOriginal', 'anoLancamento', 'anoTermino',
                                          'tempoMinutos', 'genero','notaMedia','numeroVotos','generoArtista',
                                          'personagem', 'nomeArtista', 'anoNascimento',
                                          'profissao')
df_movie_series.show(5)

+---------+---------+--------------------+--------------------+-------------+--------------+------------+------------------+---------+-----------+-------------+--------------+--------------------+-------------+--------------------+
|       id|tipoMidia|      tituloPincipal|      tituloOriginal|anoLancamento|    anoTermino|tempoMinutos|            genero|notaMedia|numeroVotos|generoArtista|    personagem|         nomeArtista|anoNascimento|           profissao|
+---------+---------+--------------------+--------------------+-------------+--------------+------------+------------------+---------+-----------+-------------+--------------+--------------------+-------------+--------------------+
|tt0036606|    movie|Another Time, Ano...|Another Time, Ano...|         1983|sem informação|         118|         Drama,War|      6.5|        327|            2|         Luigi|  Giovanni Mauriello|         1945|actor,music_depar...|
|tt0036606|    movie|Another Time, Ano...|Another Time, Ano...|         

In [ ]:
# Eliminando linhas duplicadas
df_movie_series = df_movie_series.dropDuplicates()

### Salvando o df_movie_series

In [ ]:
# Criando uma coluna para decada de lançamento
df_movie_series = df_movie_series.withColumn("decada",
                                     when((df_movie_series["anoLancamento"] >= 1970) & (df_movie_series["anoLancamento"] <= 1979), "decada_setenta")
                                     .when((df_movie_series["anoLancamento"] >= 1980) & (df_movie_series["anoLancamento"] <= 1989), "decada_oitenta")
                                     .when((df_movie_series["anoLancamento"] >= 1990) & (df_movie_series["anoLancamento"] <= 1999), "decada_noventa")
                                     .when((df_movie_series["anoLancamento"] >= 2000) & (df_movie_series["anoLancamento"] <= 2009), "dois_mil")
                                     .when((df_movie_series["anoLancamento"] >= 2010) & (df_movie_series["anoLancamento"] <= 2019), "dois_mil_dez")
                                     .otherwise("atuais"))

# Salvando o DataFrame particionado por anoNascimento
df_movie_series.write.parquet("df_movie_series.parquet", mode="overwrite", partitionBy="decada")

In [ ]:
# Apenas consultando as colunas com id's duplicado. Nesse caso tem valores duplicados
# pois para um filme pode ter mais de um artista
df_movie_series.createOrReplaceTempView("df_movie_series")

spark.sql ('''
SELECT id, COUNT(id) AS num_duplicados
FROM df_movie_series  -- Remove quotes and curly braces
GROUP BY id
HAVING COUNT(id) > 1
''').show()

+---------+--------------+
|       id|num_duplicados|
+---------+--------------+
|tt0177073|             4|
|tt0114126|             4|
|tt0088418|             4|
|tt0122319|             3|
|tt0254626|             4|
|tt0266391|             3|
|tt0102701|             3|
|tt0422998|             3|
|tt0124864|             4|
|tt0292606|             3|
|tt0108528|             2|
|tt0161292|             4|
|tt0222837|             2|
|tt0074792|             2|
|tt0365194|             3|
|tt0305962|             3|
|tt0111050|             4|
|tt0100967|             4|
|tt0110900|             2|
|tt0426097|             3|
+---------+--------------+
only showing top 20 rows



In [ ]:
# Quero observar melhor as colunas com ids duplicados
duplicados = df_movie_series.groupBy("id").count().filter("count > 1")
resultado = df_movie_series.join(duplicados, "id", "inner")
resultado.show()

+---------+---------+--------------------+--------------------+-------------+--------------+------------+--------------------+---------+-----------+-------------+--------------------+---------------+-------------+--------------------+--------------+-----+
|       id|tipoMidia|      tituloPincipal|      tituloOriginal|anoLancamento|    anoTermino|tempoMinutos|              genero|notaMedia|numeroVotos|generoArtista|          personagem|    nomeArtista|anoNascimento|           profissao|        decada|count|
+---------+---------+--------------------+--------------------+-------------+--------------+------------+--------------------+---------+-----------+-------------+--------------------+---------------+-------------+--------------------+--------------+-----+
|tt0177073|    movie|      Ormai è fatta!|      Ormai è fatta!|         1999|sem informação|         110|  Comedy,Crime,Drama|      6.5|        284|            2|    Horst Fantazzini|Stefano Accorsi|         1971|actor,writer,dire..

In [ ]:
# Fazendo uma checagem dos valores individuais
id_movie_series = df_movie_series.filter(df_movie_series['id'] == "tt0111050")
id_movie_series.show(5)

+---------+---------+--------------------+--------------------+-------------+--------------+------------+--------------------+---------+-----------+-------------+------------+-------------+-------------+--------------------+--------------+
|       id|tipoMidia|      tituloPincipal|      tituloOriginal|anoLancamento|    anoTermino|tempoMinutos|              genero|notaMedia|numeroVotos|generoArtista|  personagem|  nomeArtista|anoNascimento|           profissao|        decada|
+---------+---------+--------------------+--------------------+-------------+--------------+------------+--------------------+---------+-----------+-------------+------------+-------------+-------------+--------------------+--------------+
|tt0111050|    movie|Don't Shoot Me, I...|San gai bin fung ...|         1994|sem informação|          86|Comedy,Crime,Romance|      5.6|         29|            1|Inspector Mo|    Teresa Mo|         1960|actress,writer,pr...|decada_noventa|
|tt0111050|    movie|Don't Shoot Me, I..

### Criando um df somente para os artistas

In [ ]:
# Selecionando as colunas de interesse
df_artistas = df_movie_series.select("id", "nomeArtista", "generoArtista", "anoNascimento", "personagem", "profissao")

# Renomeando a coluna "id" para "id_filme"
df_artistas = df_artistas.withColumnRenamed("id", "idFilme")

# Criando uma nova coluna "id_artista" com valores únicos
df_artistas = df_artistas.withColumn("idArtista", monotonically_increasing_id())

# Selecionando as colunas na ordem desejada
df_artistas = df_artistas.select("idArtista", "idFilme", "nomeArtista", "generoArtista", "anoNascimento", "personagem", "profissao")

df_artistas.show(5)

+---------+---------+---------------+-------------+-------------+--------------------+--------------------+
|idArtista|  idFilme|    nomeArtista|generoArtista|anoNascimento|          personagem|           profissao|
+---------+---------+---------------+-------------+-------------+--------------------+--------------------+
|        0|tt0071327|  Wanda Neumann|            1|         1945|Wladka Stankiewic...|             actress|
|        1|tt0071562|   Diane Keaton|            1|         1946|                 Kay|actress,producer,...|
|        2|tt0074184|Isabelle Adjani|            1|         1955|               Laure|actress,soundtrac...|
|        3|tt0077168|  Sarika Thakur|            1|         1960|               Seema|actress,costume_d...|
|        4|tt0081088|    Rick Podell|            2|         1946|       Al Schmeckler|actor,writer,soun...|
+---------+---------+---------------+-------------+-------------+--------------------+--------------------+
only showing top 5 rows



In [ ]:
# Fazendo a normalização da coluna profissão no DF dos artistas
df_artistas = df_artistas.withColumn("profissao_lista", split(df_artistas["profissao"], ","))

# Encontrando o número máximo de valores separados na coluna "profissao"
max_num_valores = df_artistas.select(size("profissao_lista")).agg({"size(profissao_lista)": "max"}).collect()[0][0]

# Criando novas colunas para cada valor separado
for i in range(1, max_num_valores + 1):
    df_artistas = df_artistas.withColumn(f"profissao{i}", col("profissao_lista").getItem(i - 1))

# Preenchendo com 'não possui' quando o valor for nulo
for i in range(1, max_num_valores + 1):
    df_artistas = df_artistas.withColumn(f"profissao{i}", when(col(f"profissao{i}").isNull(), "não possui").otherwise(col(f"profissao{i}")))

# Eliminando as colunas "profissao" e "profissao_lista"
df_artistas = df_artistas.drop("profissao", "profissao_lista")

df_artistas.show(5)

+---------+---------+---------------+-------------+-------------+--------------------+----------+----------------+----------+
|idArtista|  idFilme|    nomeArtista|generoArtista|anoNascimento|          personagem|profissao1|      profissao2|profissao3|
+---------+---------+---------------+-------------+-------------+--------------------+----------+----------------+----------+
|        0|tt0071327|  Wanda Neumann|            1|         1945|Wladka Stankiewic...|   actress|      não possui|não possui|
|        1|tt0071562|   Diane Keaton|            1|         1946|                 Kay|   actress|        producer|  director|
|        2|tt0074184|Isabelle Adjani|            1|         1955|               Laure|   actress|      soundtrack|  producer|
|        3|tt0077168|  Sarika Thakur|            1|         1960|               Seema|   actress|costume_designer|não possui|
|        4|tt0081088|    Rick Podell|            2|         1946|       Al Schmeckler|     actor|          writer|soun

In [ ]:
# Apenas fazendo uma checagem na coluna profissao_3
df_filtrado = df_artistas.filter(df_artistas["profissao3"].isNotNull())

# Mostre as primeiras 5 linhas do DataFrame filtrado
df_filtrado.show(5)

+---------+---------+---------------+-------------+-------------+--------------------+----------+----------------+----------+
|idArtista|  idFilme|    nomeArtista|generoArtista|anoNascimento|          personagem|profissao1|      profissao2|profissao3|
+---------+---------+---------------+-------------+-------------+--------------------+----------+----------------+----------+
|        0|tt0071327|  Wanda Neumann|            1|         1945|Wladka Stankiewic...|   actress|      não possui|não possui|
|        1|tt0071562|   Diane Keaton|            1|         1946|                 Kay|   actress|        producer|  director|
|        2|tt0074184|Isabelle Adjani|            1|         1955|               Laure|   actress|      soundtrack|  producer|
|        3|tt0077168|  Sarika Thakur|            1|         1960|               Seema|   actress|costume_designer|não possui|
|        4|tt0081088|    Rick Podell|            2|         1946|       Al Schmeckler|     actor|          writer|soun

### Criando um DF somente para as midias

In [ ]:
df_midias = df_movie_series.select(  "id", "tipoMidia", "tituloPincipal",  "tituloOriginal",
                                     "anoLancamento", "anoTermino", "tempoMinutos", "genero", "notaMedia", "numeroVotos")

# Eliminando os valores nulos
df_midias = df_midias.dropDuplicates()
df_midias.show(5)

+---------+---------+--------------------+--------------------+-------------+--------------+------------+-------------------+---------+-----------+
|       id|tipoMidia|      tituloPincipal|      tituloOriginal|anoLancamento|    anoTermino|tempoMinutos|             genero|notaMedia|numeroVotos|
+---------+---------+--------------------+--------------------+-------------+--------------+------------+-------------------+---------+-----------+
|tt0073755|    movie|The Streetfighter...|Gyakushû! Satsuji...|         1974|sem informação|          83| Action,Crime,Drama|      6.0|       1351|
|tt0081249|    movie|       The Nude Bomb|       The Nude Bomb|         1980|sem informação|          94|Action,Comedy,Crime|      5.1|       3365|
|tt0089079|    movie|               Eleni|               Eleni|         1985|sem informação|         114|          Drama,War|      6.9|       1562|
|tt0092015|    movie| The King of Kungsan|      Stockholmsnatt|         1987|sem informação|          50|       

In [ ]:
# Normalizando a coluna genero do DF das midias
df_midias = df_midias.withColumn("genero_lista", split(df_midias["genero"], ","))

# Determine o número máximo de valores separados na coluna "profissao"
max_num_valores = df_midias.select(size("genero_lista")).agg({"size(genero_lista)": "max"}).collect()[0][0]

# Crie novas colunas para cada valor separado
for i in range(1, max_num_valores + 1):
    df_midias = df_midias.withColumn(f"genero{i}", col("genero_lista").getItem(i - 1))

# Preencher com 'não possui' quando o valor for nulo
for i in range(1, max_num_valores + 1):
    df_midias = df_midias.withColumn(f"genero{i}", when(col(f"genero{i}").isNull(), "não possui").otherwise(col(f"genero{i}")))

# Eliminando "genero" e "genero_lista"
df_midias = df_midias.drop("genero", "genero_lista")

df_midias.show(5)

+---------+---------+--------------------+--------------------+-------------+--------------+------------+---------+-----------+-------+-------+----------+
|       id|tipoMidia|      tituloPincipal|      tituloOriginal|anoLancamento|    anoTermino|tempoMinutos|notaMedia|numeroVotos|genero1|genero2|   genero3|
+---------+---------+--------------------+--------------------+-------------+--------------+------------+---------+-----------+-------+-------+----------+
|tt0073755|    movie|The Streetfighter...|Gyakushû! Satsuji...|         1974|sem informação|          83|      6.0|       1351| Action|  Crime|     Drama|
|tt0081249|    movie|       The Nude Bomb|       The Nude Bomb|         1980|sem informação|          94|      5.1|       3365| Action| Comedy|     Crime|
|tt0089079|    movie|               Eleni|               Eleni|         1985|sem informação|         114|      6.9|       1562|  Drama|    War|não possui|
|tt0092015|    movie| The King of Kungsan|      Stockholmsnatt|       

In [ ]:
# Checando a quantidade de linhas nos DF
quantidade_de_linhas_artistas = df_artistas.count()
print(f"O DataFrame artistas tem {quantidade_de_linhas_artistas} linhas.")
quantidade_de_linhas_filmes = df_midias.count()
print(f"O DataFrame midia tem {quantidade_de_linhas_filmes} linhas.")

O DataFrame artistas tem 63088 linhas.
O DataFrame midia tem 20101 linhas.


In [ ]:
# Apenas fazendo uma checagem na coluna nomeArtista
df_filtrado = df_artistas.filter(df_artistas["nomeArtista"] == 'Angelina Jolie')

# Mostre as primeiras 5 linhas do DataFrame filtrado
df_filtrado.show(5)

+---------+---------+--------------+-------------+-------------+----------+----------+----------+----------+
|idArtista|  idFilme|   nomeArtista|generoArtista|anoNascimento|personagem|profissao1|profissao2|profissao3|
+---------+---------+--------------+-------------+-------------+----------+----------+----------+----------+
|      730|tt0119906|Angelina Jolie|            1|         1975|    Claire|   actress|  producer|  director|
|     2300|tt0364045|Angelina Jolie|            1|         1975|   Illeana|   actress|  producer|  director|
|     2869|tt0113243|Angelina Jolie|            1|         1975|      Kate|   actress|  producer|  director|
|     7196|tt0356910|Angelina Jolie|            1|         1975|Jane Smith|   actress|  producer|  director|
|     7721|tt0129136|Angelina Jolie|            1|         1975|    Gloria|   actress|  producer|  director|
+---------+---------+--------------+-------------+-------------+----------+----------+----------+----------+
only showing top 5 

### Salvando o df_artista em partes segundo uma categoria de idade

In [ ]:
# Criando uma coluna para idade
df_artistas = df_artistas.withColumn("idade", 2024 - df_artistas["anoNascimento"])

# Criando a coluna de categoria de idade
df_artistas = df_artistas.withColumn("categoriaIdade",
                                     when((df_artistas["idade"] >= 1) & (df_artistas["idade"] <= 20), "juvenil")
                                     .when((df_artistas["idade"] >= 21) & (df_artistas["idade"] <= 40), "adultos")
                                     .when((df_artistas["idade"] >= 41) & (df_artistas["idade"] <= 60), "maduros")
                                     .otherwise("idosos"))

# Salvando o DataFrame particionado por anoNascimento
df_artistas.write.parquet("df_artistas.parquet", mode="overwrite", partitionBy="categoriaIdade")

In [ ]:
# Apenas vendo como ficará os arquivos
df_teste = spark.read.parquet("/content/df_artistas.parquet/categoriaIdade=idosos")
df_teste.show(5)

+----------+---------+-----------------+-------------+-------------+-----------+----------+-------------+----------+-----+
| idArtista|  idFilme|      nomeArtista|generoArtista|anoNascimento| personagem|profissao1|   profissao2|profissao3|idade|
+----------+---------+-----------------+-------------+-------------+-----------+----------+-------------+----------+-----+
|8589934592|tt0075131|      Miguel Bosé|            2|         1956|Cécil Rubes|soundtrack|        actor|  director| 68.0|
|8589934593|tt0076410|     Ornella Muti|            1|         1955|    Valérie|   actress|miscellaneous|não possui| 69.0|
|8589934594|tt0076584|Michael Pergolani|            2|         1946|       Nick|     actor|       writer|não possui| 78.0|
|8589934596|tt0080616|  Angelika Thomas|            1|         1946|      Lydia|   actress|   não possui|não possui| 78.0|
|8589934597|tt0084112|   Armand Assante|            2|         1949|Mike Hammer|     actor|     producer|soundtrack| 75.0|
+----------+----

In [ ]:
quantidade_de_linhas_artistas = df_teste.count()
print(f"O DataFrame artistas tem {quantidade_de_linhas_artistas} linhas.")

O DataFrame artistas tem 22679 linhas.


### Salvando o df_midias por década

In [ ]:
# Criando uma coluna para decada de lançamento
df_midias = df_midias.withColumn("decada",
                                     when((df_midias["anoLancamento"] >= 1970) & (df_midias["anoLancamento"] <= 1979), "setenta")
                                     .when((df_midias["anoLancamento"] >= 1980) & (df_midias["anoLancamento"] <= 1989), "oitenta")
                                     .when((df_midias["anoLancamento"] >= 1990) & (df_midias["anoLancamento"] <= 1999), "noventa")
                                     .when((df_midias["anoLancamento"] >= 2000) & (df_midias["anoLancamento"] <= 2009), "doismil")
                                     .when((df_midias["anoLancamento"] >= 2010) & (df_midias["anoLancamento"] <= 2019), "doismildez")
                                     .otherwise("atuais"))

# Salvando o DataFrame particionado por anoNascimento
df_midias.write.parquet("df_midias.parquet", mode="overwrite", partitionBy="decada")

In [ ]:
# Apenas vendo como ficará os arquivos
df_teste = spark.read.parquet("/content/df_midias.parquet/decada=doismildez")
df_teste.show(5)

+---------+---------+-------------------+-------------------+-------------+--------------+------------+---------+-----------+-------+-------+----------+
|       id|tipoMidia|     tituloPincipal|     tituloOriginal|anoLancamento|    anoTermino|tempoMinutos|notaMedia|numeroVotos|genero1|genero2|   genero3|
+---------+---------+-------------------+-------------------+-------------+--------------+------------+---------+-----------+-------+-------+----------+
|tt0376479|    movie|  American Pastoral|  American Pastoral|         2016|sem informação|         108|      6.1|      17100|  Crime|  Drama|não possui|
|tt0403935|    movie|     Action Jackson|     Action Jackson|         2014|sem informação|         144|      3.4|       3941| Action|  Crime|  Thriller|
|tt0385887|    movie|Motherless Brooklyn|Motherless Brooklyn|         2019|sem informação|         144|      6.8|      58055|  Crime|  Drama|   Mystery|
|tt0423474|    movie|      Venus & Vegas|      Venus & Vegas|         2010|sem inf

In [ ]:
quantidade_de_linhas_artistas = df_teste.count()
print(f"O DataFrame midias tem {quantidade_de_linhas_artistas} linhas.")

O DataFrame midias tem 6909 linhas.


## Tratamento DF json Artistas


In [ ]:
# Abrindo o DF pessoas coletado pela API
df_artistas_json = spark.read.json("/content/pessoas.json")
df_artistas_json.show(5)

+-----+------+-------+--------------------+--------------------+---------------+---------------+----------+--------------------+
|adult|gender|     id|           known_for|known_for_department|           name|  original_name|popularity|        profile_path|
+-----+------+-------+--------------------+--------------------+---------------+---------------+----------+--------------------+
|false|     2|  73968|[{false, /4oYBCR2...|              Acting|   Henry Cavill|   Henry Cavill|    178.94|/iWdKjMry5Pt7vmxU...|
|false|     1| 544994|[{false, /gCqS4nN...|              Acting|        Vedhika|        Vedhika|   173.902|/aVCfGUDKsycQErKw...|
|false|     2|2786960|[{false, /lntyt4O...|              Acting|Gabriel Guevara|Gabriel Guevara|   149.742|/pviRYKEEmoPUfLYw...|
|false|     1|1955840|[{false, /vh9Rk0B...|              Acting| Afra Saraçoğlu| Afra Saraçoğlu|   144.346|/qxYuc8I8q9PyxU1L...|
|false|     2|    976|[{false, /ibKeXah...|              Acting|  Jason Statham|  Jason Statham| 

In [ ]:
# Checando o esquema do json
df_artistas_json.printSchema()

root
 |-- adult: boolean (nullable = true)
 |-- gender: long (nullable = true)
 |-- id: long (nullable = true)
 |-- known_for: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- adult: boolean (nullable = true)
 |    |    |-- backdrop_path: string (nullable = true)
 |    |    |-- first_air_date: string (nullable = true)
 |    |    |-- genre_ids: array (nullable = true)
 |    |    |    |-- element: long (containsNull = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- media_type: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- origin_country: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- original_language: string (nullable = true)
 |    |    |-- original_name: string (nullable = true)
 |    |    |-- original_title: string (nullable = true)
 |    |    |-- overview: string (nullable = true)
 |    |    |-- popularity: double (nullable = true)
 |    |    

In [ ]:
# Casting do atributo id e gender para ficar no mesmo esquema que os DFs anteriores
df_artistas_json = df_artistas_json.withColumn('id', df_artistas_json['id'].cast('string'))

In [ ]:
df_artistas_json.printSchema()

root
 |-- adult: boolean (nullable = true)
 |-- gender: long (nullable = true)
 |-- id: string (nullable = true)
 |-- known_for: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- adult: boolean (nullable = true)
 |    |    |-- backdrop_path: string (nullable = true)
 |    |    |-- first_air_date: string (nullable = true)
 |    |    |-- genre_ids: array (nullable = true)
 |    |    |    |-- element: long (containsNull = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- media_type: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- origin_country: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- original_language: string (nullable = true)
 |    |    |-- original_name: string (nullable = true)
 |    |    |-- original_title: string (nullable = true)
 |    |    |-- overview: string (nullable = true)
 |    |    |-- popularity: double (nullable = true)
 |    |  

In [ ]:
# Eliminando colunas não necessárias
df_artistas_json = df_artistas_json.drop("adult", "known_for", "profile_path", 'known_for_department')

df_artistas_json.show(20)

+------+-------+--------------------+--------------------+----------+
|gender|     id|                name|       original_name|popularity|
+------+-------+--------------------+--------------------+----------+
|     2|  73968|        Henry Cavill|        Henry Cavill|    178.94|
|     1| 544994|             Vedhika|             Vedhika|   173.902|
|     2|2786960|     Gabriel Guevara|     Gabriel Guevara|   149.742|
|     1|1955840|      Afra Saraçoğlu|      Afra Saraçoğlu|   144.346|
|     2|    976|       Jason Statham|       Jason Statham|   138.873|
|     2|2318714|          Ranjan Raj|          Ranjan Raj|   128.198|
|     1|3878062|   Supaporn Malisorn|   Supaporn Malisorn|   123.835|
|     1|   1245|  Scarlett Johansson|  Scarlett Johansson|   119.322|
|     1|1397778|     Anya Taylor-Joy|     Anya Taylor-Joy|   117.192|
|     1| 224513|        Ana de Armas|        Ana de Armas|   109.767|
|     1| 115440|      Sydney Sweeney|      Sydney Sweeney|    97.889|
|     2|   1269|    

In [ ]:
# Colocando as colunas na ordem desejada
df_artistas_json = df_artistas_json.select('id', 'name','gender', 'original_name', 'popularity')
df_artistas_json.show(5)

+-------+---------------+------+---------------+----------+
|     id|           name|gender|  original_name|popularity|
+-------+---------------+------+---------------+----------+
|  73968|   Henry Cavill|     2|   Henry Cavill|    178.94|
| 544994|        Vedhika|     1|        Vedhika|   173.902|
|2786960|Gabriel Guevara|     2|Gabriel Guevara|   149.742|
|1955840| Afra Saraçoğlu|     1| Afra Saraçoğlu|   144.346|
|    976|  Jason Statham|     2|  Jason Statham|   138.873|
+-------+---------------+------+---------------+----------+
only showing top 5 rows



In [ ]:
# Checando a quantidade de linhas
quantidade_de_linhas_artistas = df_artistas_json.count()
print(f"O DataFrame artistas tem {quantidade_de_linhas_artistas} linhas.")

O DataFrame artistas tem 10000 linhas.


### Fazendo join do df_artistas com o df_artistas_json

In [ ]:
df_avaliacao_atores = df_artistas.join(df_artistas_json, df_artistas["nomeArtista"] == df_artistas_json["name"])
df_avaliacao_atores.show(20)

+---------+---------+------------------+-------------+-------------+--------------------+----------+----------------+----------------+-----+--------------+-----+------------------+------+------------------+----------+
|idArtista|  idFilme|       nomeArtista|generoArtista|anoNascimento|          personagem|profissao1|      profissao2|      profissao3|idade|categoriaIdade|   id|              name|gender|     original_name|popularity|
+---------+---------+------------------+-------------+-------------+--------------------+----------+----------------+----------------+-----+--------------+-----+------------------+------+------------------+----------+
|        1|tt0071562|      Diane Keaton|            1|         1946|                 Kay|   actress|        producer|        director| 78.0|        idosos| 3092|      Diane Keaton|     1|      Diane Keaton|    33.241|
|        2|tt0074184|   Isabelle Adjani|            1|         1955|               Laure|   actress|      soundtrack|        pro

In [ ]:
# Eliminando colunas não necessárias
df_avaliacao_atores = df_avaliacao_atores.drop('name','gender')

df_avaliacao_atores.show(20)

+---------+---------+------------------+-------------+-------------+--------------------+----------+----------------+----------------+-----+--------------+-----+------------------+----------+
|idArtista|  idFilme|       nomeArtista|generoArtista|anoNascimento|          personagem|profissao1|      profissao2|      profissao3|idade|categoriaIdade|   id|     original_name|popularity|
+---------+---------+------------------+-------------+-------------+--------------------+----------+----------------+----------------+-----+--------------+-----+------------------+----------+
|        1|tt0071562|      Diane Keaton|            1|         1946|                 Kay|   actress|        producer|        director| 78.0|        idosos| 3092|      Diane Keaton|    33.241|
|        2|tt0074184|   Isabelle Adjani|            1|         1955|               Laure|   actress|      soundtrack|        producer| 69.0|        idosos| 6553|   Isabelle Adjani|     19.28|
|        5|tt0084049|  Kathleen Quinlan|

In [ ]:
# Colocando as colunas na ordem desejada
df_avaliacao_atores = df_avaliacao_atores.select('idArtista', 'idFilme', 'id', 'nomeArtista',
                                          'original_name', 'generoArtista', 'anoNascimento', 'idade',
                                          'personagem','categoriaIdade', 'popularity', 'profissao1',
                                          'profissao2', 'profissao3')

df_avaliacao_atores.show(5)

+---------+---------+-----+----------------+----------------+-------------+-------------+-----+------------+--------------+----------+----------+----------+-------------+
|idArtista|  idFilme|   id|     nomeArtista|   original_name|generoArtista|anoNascimento|idade|  personagem|categoriaIdade|popularity|profissao1|profissao2|   profissao3|
+---------+---------+-----+----------------+----------------+-------------+-------------+-----+------------+--------------+----------+----------+----------+-------------+
|        1|tt0071562| 3092|    Diane Keaton|    Diane Keaton|            1|         1946| 78.0|         Kay|        idosos|    33.241|   actress|  producer|     director|
|        2|tt0074184| 6553| Isabelle Adjani| Isabelle Adjani|            1|         1955| 69.0|       Laure|        idosos|     19.28|   actress|soundtrack|     producer|
|        5|tt0084049| 8183|Kathleen Quinlan|Kathleen Quinlan|            1|         1954| 70.0|  Janet Dunn|        idosos|    23.417|   actress|

In [ ]:
# padronizando o nome das colunas
df_avaliacao_atores = df_avaliacao_atores.select(
                                                  col("idArtista").alias("idArtista"),
                                                  col("idFilme").alias("idFilme"),
                                                  col("id").alias("idAvaliacao"),
                                                  col("nomeArtista"),
                                                  col("original_name").alias("nomeOriginal"),
                                                  col("generoArtista"),
                                                  col("anoNascimento"),
                                                  col("idade"),
                                                  col("personagem"),
                                                  col("categoriaIdade"),
                                                  col("popularity"),
                                                  col("profissao1"),
                                                  col("profissao2"),
                                                  col("profissao3")
)
df_avaliacao_atores.show(5)

+---------+---------+-----------+----------------+----------------+-------------+-------------+-----+------------+--------------+----------+----------+----------+-------------+
|idArtista|  idFilme|idAvaliacao|     nomeArtista|    nomeOriginal|generoArtista|anoNascimento|idade|  personagem|categoriaIdade|popularity|profissao1|profissao2|   profissao3|
+---------+---------+-----------+----------------+----------------+-------------+-------------+-----+------------+--------------+----------+----------+----------+-------------+
|        1|tt0071562|       3092|    Diane Keaton|    Diane Keaton|            1|         1946| 78.0|         Kay|        idosos|    33.241|   actress|  producer|     director|
|        2|tt0074184|       6553| Isabelle Adjani| Isabelle Adjani|            1|         1955| 69.0|       Laure|        idosos|     19.28|   actress|soundtrack|     producer|
|        5|tt0084049|       8183|Kathleen Quinlan|Kathleen Quinlan|            1|         1954| 70.0|  Janet Dunn| 

In [ ]:
# Checando a quantidade de linhas
quantidade_de_linhas_join = df_avaliacao_atores.count()
print(f"O DataFrame artistas tem {quantidade_de_linhas_join} linhas.")

O DataFrame artistas tem 17317 linhas.


In [ ]:
# Fazendo uma checagem dos valores individuais
filtro_atores = df_avaliacao_atores.filter(df_movie_series['nomeArtista'] == "Robert Downey Jr.")
filtro_atores.show(5)

+---------+---------+-----------+-----------------+-----------------+-------------+-------------+-----+--------------------+--------------+----------+----------+----------+----------+
|idArtista|  idFilme|idAvaliacao|      nomeArtista|     nomeOriginal|generoArtista|anoNascimento|idade|          personagem|categoriaIdade|popularity|profissao1|profissao2|profissao3|
+---------+---------+-----------+-----------------+-----------------+-------------+-------------+-----+--------------------+--------------+----------+----------+----------+----------+
|     1595|tt0373469|       3223|Robert Downey Jr.|Robert Downey Jr.|            2|         1965| 59.0|      Harry Lockhart|       maduros|     47.03|     actor|  producer|soundtrack|
|     3124|tt0119196|       3223|Robert Downey Jr.|Robert Downey Jr.|            2|         1965| 59.0|          Clyde Pell|       maduros|     47.03|     actor|  producer|soundtrack|
|     5312|tt0314676|       3223|Robert Downey Jr.|Robert Downey Jr.|           

### Salvando o df com os joins

In [ ]:
# Salvando o DataFrame particionado por anoNascimento
df_avaliacao_atores.write.parquet("df_avaliacao_atores.parquet", mode="overwrite", partitionBy="categoriaIdade")

In [ ]:
# Apenas vendo como ficará os arquivos
df_teste = spark.read.parquet("/content/df_avaliacao_atores.parquet/categoriaIdade=maduros")
df_teste.show(20)

+---------+---------+-----------+-----------------+-----------------+-------------+-------------+-----+--------------------+----------+----------+----------------+------------------+
|idArtista|  idFilme|idAvaliacao|      nomeArtista|     nomeOriginal|generoArtista|anoNascimento|idade|          personagem|popularity|profissao1|      profissao2|        profissao3|
+---------+---------+-----------+-----------------+-----------------+-------------+-------------+-----+--------------------+----------+----------+----------------+------------------+
|       17|tt0097332|      69597|   Drew Barrymore|   Drew Barrymore|            1|         1975| 49.0|          Joleen Cox|    30.283|  producer|         actress|        soundtrack|
|       29|tt0108282|      12672|       Carina Lau|       Carina Lau|            1|         1964| 60.0|         Ching-Ching|     13.93|   actress|        producer|        não possui|
|       39|tt0113501|      56871|  Blair Underwood|  Blair Underwood|            2|  